# Notebook 10 — The 3-layer cache and `CACHE_VERSION` invalidation

**Purpose:** Make repeated work near-free. Build the §7.6 cache layers and prove the
`CACHE_VERSION` discipline catches the silent-stale-cache bug before a careless prompt
edit bleeds into a thousand downstream answers.

**Exam relevance:** API & SDK Usage (Anthropic prompt cache), cost engineering.

**Design refs:** `docs/marginalia-design.md` §7.6.

**Depends on:** NB 02 (`analyze_source` is the cacheable step), the engine's
`engine/utils/cost_tracker.py` (the `cached: bool` field is already wired).

**The three layers:**

| Layer | What it caches | Where it lives | Bust on |
|---|---|---|---|
| **L1** | `SourceAnalysis` outputs (the cacheable analyze step) | `<wiki>/.wiki/cache/analysis/*.json` | `content_sha`, `CACHE_VERSION`, `prompt.name@version` |
| **L2** | Raw responses for any deterministic LLM call | `<wiki>/.wiki/cache/llm/*.json` | `sha256(rendered_prompt)`, `model_id`, `CACHE_VERSION` |
| **L3** | Anthropic's native prompt cache on the system block | server-side (5-min TTL) | byte-identical system prompt prefix |

**The CACHE_VERSION rule:** edit a prompt body → bump `prompt.version` in its frontmatter.
Edit something that crosses prompts (a Pydantic schema, a model swap, a serialization
change) → bump the global `CACHE_VERSION` in `engine/cache/__init__.py`. This notebook
demonstrates both knobs and the failure mode of skipping them.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import shutil
import time
from pathlib import Path

from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table

console = Console()

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set in .env"

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

POC_WIKI = REPO / "notebooks" / "data" / "poc-wiki"
NB_TMP = Path("/tmp/marginalia-nb10")
WIKI_ROOT = NB_TMP / "wiki"
CACHE_ROOT = WIKI_ROOT / ".wiki" / "cache"

# Fresh slate per run.
if NB_TMP.exists():
    shutil.rmtree(NB_TMP)
shutil.copytree(POC_WIKI, WIKI_ROOT)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(f"[dim]wiki_root = [cyan]{WIKI_ROOT}[/cyan][/dim]")
console.print(f"[dim]cache_root = [cyan]{CACHE_ROOT}[/cyan][/dim]")

In [ ]:
from anthropic import Anthropic

from engine.agents.ingest.analyze import (
    PROMPT_NAME,
    analyze_source,
    compute_content_sha256,
)
from engine.cache import CACHE_VERSION
from engine.cache.admin import clear_all, gc, stats
from engine.cache.decorators import cached_analyze, cached_llm_call
from engine.cache.l1_analysis import AnalysisCache
from engine.cache.l2_responses import CachedLLMResponse, ResponseCache
from engine.models.pages import SourceKind
from engine.models.wiki_config import MarginaliaConfig
from engine.prompts import load_prompt
from engine.utils.cost_tracker import record_attempt

# Anthropic's prompt cache has a per-model minimum cacheable-prefix size
# (empirically ~2-3K tokens for Haiku/Sonnet 4.x; below that, caching is
# silently skipped). The PoC wiki's purpose.md + AGENTS.md by themselves
# fall under, so we pad the sandbox copy to clear the threshold. Production
# wikis with real bodies typically don't need this.
_topics = [
    "the Apollo launch programme and its inter-team coordination",
    "Q2 pricing experiments and the resulting commercial decisions",
    "platform reliability runbooks for on-call engineers",
    "data ingest pipelines including upstream contracts",
    "model routing policies and per-model cost tradeoffs",
    "incident response practices and post-mortem rituals",
    "vendor selection trade-offs across infra and SaaS choices",
    "security review cadences and threat-modeling templates",
    "team rituals, ceremonies, and collaboration patterns",
    "long-running technical debts and pay-down strategies",
]
_pad_lines = ["", "## Extended scope (stable boilerplate; pads L3 prefix)"]
for i, area in enumerate(_topics * 5):  # 50 lines, ~150 tokens each
    _pad_lines.append(
        f"- Topic {i:03d}: this wiki covers internal product engineering "
        f"knowledge around {area}, including architecture decisions, "
        f"recurring discussion threads, decision records, retrospectives, "
        f"meeting summaries, and the long-form analyses that stitch them "
        f"together with stable terminology, citations, and cross-references."
    )
purpose_path = WIKI_ROOT / "purpose.md"
purpose_path.write_text(purpose_path.read_text() + "\n".join(_pad_lines), encoding="utf-8")

config = MarginaliaConfig.load(WIKI_ROOT)
client = Anthropic()

L1 = AnalysisCache(CACHE_ROOT / "analysis")
L2 = ResponseCache(CACHE_ROOT / "llm")

console.print(f"[bold]CACHE_VERSION = {CACHE_VERSION!r}[/bold]")
console.print(f"[dim]L1 root: {L1.root}[/dim]")
console.print(f"[dim]L2 root: {L2.root}[/dim]")

## Part A — L1 caches the analyze step

L1 is the highest-leverage layer. The §7.3 ingest split was designed precisely so
the *analyze* step is pure over `(content, prompt.version)` — same inputs always
produce the same `SourceAnalysis`. Wrapping it in `cached_analyze` means re-runs
on unchanged sources never call Anthropic at all.

In [ ]:
fixture_a = (POC_WIKI / "raw" / "good_source.md").read_text(encoding="utf-8")
console.print(f"[dim]fixture A — sha256: {compute_content_sha256(fixture_a)[:16]}…[/dim]")
console.print(fixture_a)

In [ ]:
t0 = time.perf_counter()
analysis_1, hit_1 = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=L1, client=client,
)
t_1 = time.perf_counter() - t0

console.print(f"[bold]first call:[/bold] cache hit = [red]{hit_1}[/red], "
              f"latency = {t_1*1000:.0f} ms")
console.print(f"  proposed_title: {analysis_1.proposed_title}")
console.print(f"  entities:       {analysis_1.entities}")

In [ ]:
t0 = time.perf_counter()
analysis_2, hit_2 = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=L1, client=client,
)
t_2 = time.perf_counter() - t0

console.print(f"[bold]second call:[/bold] cache hit = [green]{hit_2}[/green], "
              f"latency = {t_2*1000:.2f} ms")
assert hit_2 is True, "L1 must hit on identical input"
assert analysis_2.proposed_title == analysis_1.proposed_title

speedup = t_1 / max(t_2, 1e-9)
console.print(f"[dim]speedup: {speedup:.0f}× (API call avoided)[/dim]")

In [ ]:
# Inspect the on-disk entry — keys, envelopes, no surprises.
entries = sorted((L1.root).glob("*.json"))
assert len(entries) == 1, f"expected 1 L1 entry, got {len(entries)}"
payload = json.loads(entries[0].read_text(encoding="utf-8"))

console.print(f"[bold]L1 entry:[/bold] {entries[0].name}")
console.print(f"  cache_version: {payload['cache_version']}")
console.print(f"  cached_at:     {payload['cached_at']}")
console.print(f"  proposed_type: {payload['analysis']['proposed_type']}")

## Part B — L2 caches arbitrary deterministic LLM calls

L1 caches at the function level; L2 caches at the call level. Useful for the
not-yet-built deterministic prompts (lint contradiction probes, scaffold prompts)
that don't fit L1's strict-schema shape but still benefit from "same inputs →
same answer" reuse. The key folds in `model_id` so a Haiku→Sonnet swap busts
the cache automatically.

In [ ]:
async def real_haiku_call() -> CachedLLMResponse:
    # One-shot deterministic call — Haiku, temperature=0, fixed system + user.
    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=80,
        temperature=0,
        system="Reply with the single word PING.",
        messages=[{"role": "user", "content": "ping?"}],
    )
    return CachedLLMResponse(
        text=resp.content[0].text,
        model="claude-haiku-4-5",
        tokens_in=resp.usage.input_tokens,
        tokens_out=resp.usage.output_tokens,
    )

rendered = "system:Reply with the single word PING.\nuser:ping?"

t0 = time.perf_counter()
r1, hit_1 = await cached_llm_call(
    cache=L2, rendered_prompt=rendered, model="claude-haiku-4-5", fn=real_haiku_call,
)
t_l2_miss = time.perf_counter() - t0

t0 = time.perf_counter()
r2, hit_2 = await cached_llm_call(
    cache=L2, rendered_prompt=rendered, model="claude-haiku-4-5", fn=real_haiku_call,
)
t_l2_hit = time.perf_counter() - t0

console.print(f"L2 first call:  hit = [red]{hit_1}[/red], {t_l2_miss*1000:.0f} ms, text={r1.text!r}")
console.print(f"L2 second call: hit = [green]{hit_2}[/green], {t_l2_hit*1000:.2f} ms, text={r2.text!r}")
assert hit_2 is True
assert r2.text == r1.text

In [ ]:
# Switch model — same prompt, different cache key. Should miss.
async def real_sonnet_call() -> CachedLLMResponse:
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=80,
        temperature=0,
        system="Reply with the single word PING.",
        messages=[{"role": "user", "content": "ping?"}],
    )
    return CachedLLMResponse(
        text=resp.content[0].text,
        model="claude-sonnet-4-6",
        tokens_in=resp.usage.input_tokens,
        tokens_out=resp.usage.output_tokens,
    )

r3, hit_3 = await cached_llm_call(
    cache=L2, rendered_prompt=rendered, model="claude-sonnet-4-6", fn=real_sonnet_call,
)
console.print(f"Sonnet on same prompt: hit = [red]{hit_3}[/red] (model_id is in the key)")
assert hit_3 is False

## Part C — L3 is Anthropic's native prompt cache

Wired into `build_analyze_system_blocks` (production default `prompt_cache=True`).
The wiki-config tail of the system prompt — `purpose.md` + `AGENTS.md` — is marked
`cache_control: {"type": "ephemeral"}`. First call pays a small cache-creation
premium; every subsequent call within the 5-minute TTL reads it back at a deep
discount, even when the user message (the source content) is different.

Rule of thumb: L1 saves on cache *hits*; L3 saves on cache *misses* — they are
complements, not alternatives.

In [ ]:
# Use a fresh content body so L1 misses (we want to see the API call's L3 numbers).
# Note: Part A's first cell already issued an analyze call with the same system
# prompt, so L3 was created server-side at that point. This call is the *first
# observation* of L3 reads, but not the first contributor to cache_creation.
fixture_b = (POC_WIKI / "raw" / "ambiguous_source.md").read_text(encoding="utf-8")

usage_b: dict = {}
await analyze_source(
    fixture_b, SourceKind.LOCAL_FILE, config,
    client=client,
    out_usage=usage_b,
)
console.print("[bold]analyze on fixture B (system prompt already cached by Part A):[/bold]")
for k, v in usage_b.items():
    console.print(f"  {k}: {v}")

In [ ]:
# A second different content — confirms L3 keeps reading the same cached prefix
# regardless of how the user message changes.
fixture_c = (POC_WIKI / "raw" / "good_source.md").read_text(encoding="utf-8") + "\n\nExtra paragraph to bust L1.\n"

usage_c: dict = {}
await analyze_source(
    fixture_c, SourceKind.LOCAL_FILE, config,
    client=client,
    out_usage=usage_c,
)
console.print("[bold]analyze on fixture C (different content, same cached prefix):[/bold]")
for k, v in usage_c.items():
    console.print(f"  {k}: {v}")

l3_table = Table(title="L3 prompt cache effect")
l3_table.add_column("call")
l3_table.add_column("input_tokens", justify="right")
l3_table.add_column("cache_creation", justify="right")
l3_table.add_column("cache_read", justify="right")
l3_table.add_row("fixture B", str(usage_b['input_tokens']),
                 str(usage_b['cache_creation_input_tokens']),
                 str(usage_b['cache_read_input_tokens']))
l3_table.add_row("fixture C", str(usage_c['input_tokens']),
                 str(usage_c['cache_creation_input_tokens']),
                 str(usage_c['cache_read_input_tokens']))
console.print(l3_table)

# Anthropic only caches prefixes that exceed the per-model minimum
# (empirically ~2-3K tokens for Haiku/Sonnet 4.x; below that, caching is
# silently skipped). The padding step in the setup cell pushes the prefix
# above the threshold.
if usage_c['cache_read_input_tokens'] > 0:
    saved = usage_c['cache_read_input_tokens']
    full_input = saved + usage_c['input_tokens']
    pct = 100 * saved / full_input
    console.print(f"[bold green]✓ L3 fired:[/bold green] warm call read "
                  f"{saved} tokens from server cache "
                  f"(~{pct:.0f}% of total input — billed at ~10% of normal).")
else:
    console.print("[yellow]⚠ L3 did not fire — cacheable prefix likely under the per-model minimum.[/yellow]")

## Part D — The silent-stale-cache bug

The single most insidious failure mode in cached LLM systems: edit a prompt's body
without bumping a version. The cache key is unchanged, the cache hits on stale
output, every downstream answer reflects yesterday's prompt. No error, no warning.

We reproduce it against a *copy* of the production prompt (so we never dirty
`engine/prompts/`) and demonstrate both fixes: bump `prompt.version` (per-prompt
lever) and bump `CACHE_VERSION` (global escape hatch).

In [ ]:
# Stage a working copy of the prompt under tmp.
PROMPTS_TMP = NB_TMP / "prompts"
PROMPTS_TMP.mkdir(exist_ok=True)
real_prompt_path = REPO / "engine" / "prompts" / f"{PROMPT_NAME}.md"
demo_prompt_path = PROMPTS_TMP / f"{PROMPT_NAME}.md"
shutil.copy(real_prompt_path, demo_prompt_path)

# Fresh L1 sandbox for this demo so we can read off counts cleanly.
DEMO_L1_ROOT = CACHE_ROOT / "demo_d_analysis"
DEMO_L1_ROOT.mkdir(exist_ok=True)
demo_cache = AnalysisCache(DEMO_L1_ROOT)

# 1) Prime the cache with prompt v1 against fixture A.
prompt_v1 = load_prompt(PROMPT_NAME, prompts_dir=PROMPTS_TMP)
console.print(f"[bold]prompt as loaded:[/bold] {prompt_v1.name}@{prompt_v1.version}")

primed, hit_p = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=demo_cache, client=client, prompt=prompt_v1,
)
assert hit_p is False
title_v1 = primed.proposed_title
console.print(f"[dim]primed entry — title: {title_v1!r}[/dim]")

In [ ]:
# 2) THE BUG: edit the prompt body without bumping the frontmatter version.
text = demo_prompt_path.read_text(encoding="utf-8")
patched = text.replace(
    "You are the Marginalia ingest analyzer.",
    "You are the Marginalia ingest analyzer.\nALWAYS prefix proposed_title with 'EDITED:'.",
)
assert patched != text, "prompt edit must change the body"
demo_prompt_path.write_text(patched, encoding="utf-8")

# Reload — version field is still v1 in the frontmatter.
prompt_v1_edited = load_prompt(PROMPT_NAME, prompts_dir=PROMPTS_TMP)
console.print(f"[bold]reloaded prompt:[/bold] {prompt_v1_edited.name}@{prompt_v1_edited.version}")
console.print("[yellow]body changed, version unchanged — the cache key is identical[/yellow]")

stale, hit_s = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=demo_cache, client=client, prompt=prompt_v1_edited,
)
console.print(f"\n[bold red]bug demo:[/bold red] cache hit = {hit_s}")
console.print(f"  served title:  {stale.proposed_title!r}")
console.print(f"  expected new:  starts with 'EDITED:'")
console.print(f"  was primed as: {title_v1!r}")
assert hit_s is True
assert stale.proposed_title == title_v1, "stale entry served — same as primed"
assert not stale.proposed_title.startswith("EDITED:"), "the new prompt was bypassed"

In [ ]:
# 3) FIX (a) — bump prompt.version. Per-prompt lever; cleanest for one-prompt edits.
patched_v2 = patched.replace("version: v1", "version: v2")
assert "version: v2" in patched_v2
demo_prompt_path.write_text(patched_v2, encoding="utf-8")

prompt_v2 = load_prompt(PROMPT_NAME, prompts_dir=PROMPTS_TMP)
console.print(f"[bold]bumped prompt:[/bold] {prompt_v2.name}@{prompt_v2.version}")

fresh, hit_f = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=demo_cache, client=client, prompt=prompt_v2,
)
console.print(f"\n[bold green]fix (a):[/bold green] cache hit = {hit_f}")
console.print(f"  served title: {fresh.proposed_title!r}")
assert hit_f is False, "version bump must produce a miss"
assert fresh.proposed_title.startswith("EDITED:"), "fresh call should reflect the new prompt"

In [ ]:
# 4) FIX (b) — for changes that cross multiple prompts (Pydantic schema, model swap, serialization),
#    bump the global CACHE_VERSION constant. We can't actually edit the constant in
#    the running kernel, but we can demonstrate the same effect by clearing the cache:
#    bumping CACHE_VERSION simply makes every existing entry's key unreachable, which
#    is operationally equivalent to clearing.

before = stats(CACHE_ROOT)
console.print("[bold]before global bust:[/bold]")
for layer in before.layers:
    console.print(f"  {layer.layer}: {layer.entries} entries")

removed = clear_all(CACHE_ROOT)
console.print(f"[dim]\\nclear_all() removed {removed} entr{'y' if removed == 1 else 'ies'} "
              f"— same observable effect as a CACHE_VERSION bump.[/dim]")

after = stats(CACHE_ROOT)
console.print("\\n[bold]after:[/bold]")
for layer in after.layers:
    console.print(f"  {layer.layer}: {layer.entries} entries")

## Part E — Admin commands

`marginalia cache stats`, `gc`, `clear` mirror the `marginalia jobs` shape from
NB 09: short-lived process, JSON-serializable output where useful, confirmation
on destructive verbs. Re-prime a few entries and exercise each verb.

In [ ]:
# Re-seed L1 + L2 with one entry each so the admin verbs have data.
await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=L1, client=client,
)
await cached_llm_call(
    cache=L2, rendered_prompt=rendered, model="claude-haiku-4-5", fn=real_haiku_call,
)

s = stats(CACHE_ROOT)
table = Table(title=f"cache stats — {s.root}")
table.add_column("layer")
table.add_column("entries", justify="right")
table.add_column("bytes", justify="right")
table.add_column("oldest")
for layer in s.layers:
    table.add_row(layer.layer, str(layer.entries), str(layer.bytes),
                  layer.oldest.strftime("%Y-%m-%d %H:%M:%S") if layer.oldest else "-")
console.print(table)

In [ ]:
# gc with --older-than 0 — drop entries older than "right now".
# Sleep half a second so freshly-written entries are eligible.
import time
time.sleep(0.6)
removed = gc(CACHE_ROOT, age_days=0)
console.print(f"[green]gc --older-than 0:[/green] removed {removed} entr{'y' if removed == 1 else 'ies'}")

In [ ]:
# CLI smoke: exercise the bound verbs through the real Typer app.
import subprocess

# Re-seed so the CLI has something to show.
await cached_analyze(fixture_a, SourceKind.LOCAL_FILE, config, cache=L1, client=client)

env = os.environ.copy()
env["MARGINALIA_CACHE_ROOT"] = str(CACHE_ROOT)

result = subprocess.run(
    ["uv", "run", "marginalia", "cache", "stats"],
    cwd=REPO, env=env, capture_output=True, text=True,
)
print(result.stdout)

## Part F — Cost integration: cached attempts still emit `CostRecord` rows

The `cached: bool` column on `CostRecord` was added in NB 02 specifically for this:
NB 11 will count `WHERE cached = 1` to report savings. NB 10 only needs to confirm
that a cache-hit path produces a record with `cached=True` — even though no tokens
were spent.

In [ ]:
# Re-run analyze with L1 hot — emulate a worker-style cost log.
analysis_hit, hit = await cached_analyze(
    fixture_a, SourceKind.LOCAL_FILE, config,
    cache=L1, client=client,
)
assert hit is True

# Record what NB 11's audit DB would see.
record = record_attempt(
    agent="ingest",
    model="claude-haiku-4-5",
    tokens_in=0, tokens_out=0,  # no API call, no tokens
    cached=True,
)
console.print(f"[bold]CostRecord on cache hit:[/bold]")
console.print(f"  agent:     {record.agent}")
console.print(f"  model:     {record.model}")
console.print(f"  tokens:    {record.tokens_in} in, {record.tokens_out} out")
console.print(f"  cost_usd:  ${record.cost_usd:.6f}")
console.print(f"  cached:    {record.cached}")

In [ ]:
# Headline number: 5-source ingest before vs. after the cache.
fixtures = [
    ("good_source",    fixture_a),
    ("ambiguous",      fixture_b),
    ("good_plus",      fixture_c),
    ("good_alt_a",     fixture_a + "\n\nAlternate paragraph A.\n"),
    ("good_alt_b",     fixture_a + "\n\nAlternate paragraph B.\n"),
]

# Wipe L1 so the first pass is a clean cold start.
clear_all(CACHE_ROOT)

cold_t0 = time.perf_counter()
for name, body in fixtures:
    await cached_analyze(body, SourceKind.LOCAL_FILE, config, cache=L1, client=client)
cold_total = time.perf_counter() - cold_t0

warm_t0 = time.perf_counter()
warm_hits = 0
for name, body in fixtures:
    _, h = await cached_analyze(body, SourceKind.LOCAL_FILE, config, cache=L1, client=client)
    if h:
        warm_hits += 1
warm_total = time.perf_counter() - warm_t0

delta = Table(title="5-source ingest: cold vs. warm L1")
delta.add_column("pass")
delta.add_column("total time", justify="right")
delta.add_column("hits", justify="right")
delta.add_column("avg/source", justify="right")
delta.add_row("cold (cache empty)", f"{cold_total:.2f} s", "0/5", f"{cold_total/5*1000:.0f} ms")
delta.add_row("warm (cache full)",  f"{warm_total:.3f} s", f"{warm_hits}/5", f"{warm_total/5*1000:.1f} ms")
console.print(delta)

# Record numbers for receipts.
NB10_NUMBERS = {
    "cold_total_s": cold_total,
    "warm_total_s": warm_total,
    "warm_hits": warm_hits,
    "speedup": cold_total / max(warm_total, 1e-9),
    "l3_fixture_b_cache_creation": usage_b.get("cache_creation_input_tokens", 0),
    "l3_fixture_b_cache_read":     usage_b.get("cache_read_input_tokens", 0),
    "l3_fixture_c_cache_creation": usage_c.get("cache_creation_input_tokens", 0),
    "l3_fixture_c_cache_read":     usage_c.get("cache_read_input_tokens", 0),
}
console.print(f"[dim]speedup: {NB10_NUMBERS['speedup']:.0f}× on the warm pass[/dim]")

## Part G — Receipts

Numbers from this run feed `engine/decisions/cache.md` — same pattern as
`engine/decisions/job-queue.md` from NB 09.

In [ ]:
from datetime import date

receipts = f"""# Cache layers receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/10_cache_layers.ipynb`
**Design refs:** `docs/marginalia-design.md` §7.6 (caching), §13.2
(`cost_records.cached` consumes the hit flag set here).

## Architecture

```
              analyze_source(content, source_kind, config, ...)
                                │
                                ▼
              ┌──────────────────────────────────────────┐
              │  cached_analyze (engine/cache/decorators)│
              │  key = sha256(                           │
              │    content_sha | CACHE_VERSION |         │
              │    prompt.name@prompt.version            │
              │  )                                       │
              └─────────────┬────────────────────────────┘
                            │ miss                 │ hit
                            ▼                       ▼
              ┌────────────────────────┐    return cached
              │  Anthropic.messages.   │    SourceAnalysis
              │  create(system=[       │    (no API call)
              │    {{                  │
              │      "type": "text",   │
              │      "text": ...,      │
              │      "cache_control":  │
              │        {{"type":       │
              │         "ephemeral"}}  │
              │    }}                  │ ◄── L3 (server-side)
              │  ], ...)               │     5-min TTL
              └────────────────────────┘
```

## Layer composition

| Layer | Storage | Key | TTL |
|---|---|---|---|
| L1 | `<wiki>/.wiki/cache/analysis/<key>.json` | `sha256(content_sha \\| CACHE_VERSION \\| prompt.name@version)` | unbounded; GC by `cached_at` |
| L2 | `<wiki>/.wiki/cache/llm/<key>.json` | `sha256(sha256(rendered_prompt) \\| model_id \\| CACHE_VERSION)` | unbounded; GC by `cached_at` |
| L3 | Anthropic-server | implicit on byte-identical system block | 5 min (ephemeral) |

## Headline numbers (this run)

| Measurement | Value |
|---|---|
| 5-source ingest, cold L1 | {NB10_NUMBERS['cold_total_s']:.2f} s |
| 5-source ingest, warm L1 | {NB10_NUMBERS['warm_total_s']:.3f} s |
| Warm-pass cache hits | {NB10_NUMBERS['warm_hits']}/5 |
| Speedup on warm pass | ~{NB10_NUMBERS['speedup']:.0f}× |
| L3 cache_read tokens, fixture B | {NB10_NUMBERS['l3_fixture_b_cache_read']} |
| L3 cache_read tokens, fixture C | {NB10_NUMBERS['l3_fixture_c_cache_read']} |

The L3 row is the proof that the prompt cache fired against the live API.
A non-zero `cache_read` means Anthropic served the cacheable prefix from its
server-side cache at ~10% the normal input cost. If both rows are 0, the
cacheable prefix fell below the per-model minimum (empirically ~2-3K tokens
for Haiku/Sonnet 4.x) — silent-skip, not an error. NB 10 pads `purpose.md`
in the sandbox to clear the threshold; production wikis with real bodies
typically don't need padding.

## Design decisions

| Decision | Choice | Why |
|---|---|---|
| L1 key spine | `content_sha + CACHE_VERSION + prompt.name@prompt.version` | Editing a prompt's body without bumping its frontmatter `version` produces a stale hit — that's the demonstrated bug. The two version knobs reconcile both bullets in design §7.6 |
| L2 key spine | `sha256(rendered_prompt) + model_id + CACHE_VERSION` | The rendered prompt already contains the full system+user text, so body changes invalidate naturally without a separate version field |
| L3 cache breakpoint | tail of the system block | The wiki-config tail (`purpose.md` + `AGENTS.md`) is large and stable; one breakpoint covers prompt.system + config in a single cached prefix |
| Atomic writes | tempfile-then-`os.replace` | Two workers racing on the same key produce one valid file. No advisory locks needed |
| Storage location | `<wiki_root>/.wiki/cache/{{analysis,llm}}/<key>.json` | Per-wiki, gitignored, sits next to `jobs.db` |
| L2 wrapping `synthesize_page` | **out of scope** | `synthesize_page` is state-dependent on `existing_pages`; not a clean L2 candidate |

## CACHE_VERSION discipline

Two version knobs, two scopes:

| Edit | What to bump |
|---|---|
| One prompt's body or examples | `version` field in that prompt's frontmatter |
| Pydantic schema, serialization, model default | `CACHE_VERSION` in `engine/cache/__init__.py` |

The bug demo (Part D) shows what happens when neither is bumped: the cache
hits stale, every downstream answer reflects yesterday's prompt, no error.

## CLI

```text
marginalia cache stats              # entries, bytes, oldest/newest per layer
marginalia cache gc --older-than 7  # drop entries older than N days (float OK)
marginalia cache clear --yes        # wipe L1 + L2 (L3 expires server-side)
```

Resolution order for `--root`: explicit flag → `$MARGINALIA_CACHE_ROOT`
→ `$WIKI_CONTENT_REPO/.wiki/cache` → `./.wiki/cache`.
"""

receipts_path = REPO / "engine" / "decisions" / "cache.md"
receipts_path.write_text(receipts, encoding="utf-8")
console.print(f"[green]wrote receipts:[/green] {receipts_path.relative_to(REPO)}")
console.print(f"[dim]{len(receipts)} bytes, {receipts.count(chr(10))} lines[/dim]")

## What to extract

| Notebook artifact | Lands at |
|---|---|
| `CACHE_VERSION` constant + public exports | `engine/cache/__init__.py` |
| Pure key builders (L1, L2) | `engine/cache/keys.py` |
| `AnalysisCache`, `L1Entry` | `engine/cache/l1_analysis.py` |
| `ResponseCache`, `CachedLLMResponse`, `L2Entry` | `engine/cache/l2_responses.py` |
| `cached_analyze`, `cached_llm_call` | `engine/cache/decorators.py` |
| `stats`, `gc`, `clear_all`, `CacheStats`, `LayerStats` | `engine/cache/admin.py` |
| `marginalia cache stats|gc|clear` | `engine/cli/cache.py` (registered in `engine/cli/main.py`) |
| L3 wiring (`build_analyze_system_blocks`, `prompt_cache=True` default, `out_usage` dict) | `engine/agents/ingest/analyze.py` |
| Tests (key determinism, L1 hit/miss/version-bust, L2 hit/miss/model-id, concurrent writes) | `tests/test_cache_keys.py`, `tests/test_cache_l1.py`, `tests/test_cache_l2.py` |
| Receipts | `engine/decisions/cache.md` |

**Out of scope (deliberate):** distributed cache (Redis), L2 wrapping of
`synthesize_page` (state-dependent), audit-DB persistence of `cost_records`
(NB 11). NB 10 confirms the in-memory shape; NB 11 will write rows.